# Data Preparation 

Building the clean pool of retracted biomedical papers used as the retracted-source
side of the matched-pair benchmark.

**Source:** Retraction Watch database (Crossref/GitLab), accessed 2026-09-03.
Database updates daily, so counts are specific to this date.

**Filtering funnel:**
| Stage | Count |
|---|---|
| Raw records | 72,360 |
| True retractions (drop corrections, expressions of concern, reinstatements) | 66,727 |
| Biomedical subjects — (HSC) + (BLS) | 34,019 |
| With usable original-paper PubMed ID | 26,413 |
| **Tier-1: confirmed content failure** (fabrication, data/image error, unreliable results, misconduct) | **14,984** |

**Tier-1 reason set** (exact match, not substring): Falsification/Fabrication of Data,
Error in Data, Unreliable Data, Unreliable Results and/or Conclusions,
Manipulation of Images, Original Data and/or Images not Provided, Misconduct by Author,
Misconduct - Official Investigation(s)/Finding(s).

Process/suspicion reasons (Investigation by X, Concerns about X) and administrative ones
(Duplication, Author Unresponsive) are **excluded**, but the findings there may still be valid.

**Output:** `data/interim/tier1_pool.csv` (14,984 papers; metadata + PMIDs, abstracts fetched fresh in Step 2).
Abstract resolution ~98% on a 50-paper sample (misses are old case reports with no abstract).

In [1]:
# sanity check

import sys, pandas
print(sys.executable)
print(pandas.__version__)

c:\Users\kezia\Documents\retraction-aware-citeeval\.venv\Scripts\python.exe
3.0.5


In [3]:
# importing pands 
import pandas as pd 

# reading the retraction watch csv file into a dataframe 
# dtype = str is used to make sure every column is loaded as text 
# keep_default_na to ensure blank cells are left empty

df = pd.read_csv('../data/raw/retraction_watch.csv', dtype = str, keep_default_na = False)

# print rows (retraction recrods)
print('rows:', len(df))

# print column names to ensure the file matches retraction watch
print('columns:', list(df.columns))

rows: 72360
columns: ['Record ID', 'Title', 'Subject', 'Institution', 'Journal', 'Publisher', 'Country', 'Author', 'URLS', 'ArticleType', 'RetractionDate', 'RetractionDOI', 'RetractionPubMedID', 'OriginalPaperDate', 'OriginalPaperDOI', 'OriginalPaperPubMedID', 'RetractionNature', 'Reason', 'Paywalled', 'Notes', 'Unnamed: 20']


In [4]:
# print the first 3 rows to check data, additional code for formatting
with pd.option_context('display.max_colwidth', 200):
    display(df.head(3).T)

,0,1,2
Record ID,72857,72831,72830
Title,An innovative approach to study human-snake interactions: insights into tourists’ perception of vipers and their ability to detect them,Medical wireless sensor diagnosis and children's respiratory tract infection nursing intervention,Internet of things smart medical system and nursing intervention of glucocorticoid drug use
Subject,(SOC) Psychology;(SOC) Sports and Recreation;,(B/T) Technology;(HSC) Medicine - Nursing;(HSC) Medicine - Otorhinolaryngology;(HSC) Medicine - Pediatrics;,(B/T) Technology;(HSC) Medicine - General;(HSC) Medicine - Nursing;
Institution,"Department of Ecosystem Research, Ecological Risk and Conservation Biology, Institute of Biodiversity and Ecosystem Research, Bulgarian Academy of Sciences, Sofia, Bulgaria;","Zaozhuang Maternal and Child Health Hospital, Zaozhuang, Shandong, 277100, China; Department of Pediatrics, Zaozhuang Maternal and Child Health Hospital, Zaozhuang, Shandong, 277100, China;","Department of Pediatrics, Linyi Central Hospital, Linyi, Shandong, 276000, China; Department of Cardiology, Linyi Third People's Hospital, Linyi, Shandong, 276000, China; Oncology Department, Zibo..."
Journal,PeerJ,Microprocessors and Microsystems,Microprocessors and Microsystems
Publisher,PeerJ,Elsevier,Elsevier
Country,Bulgaria,China,China
Author,Angel Valentinov Dyugmedzhiev,Ling Zhang;Cunfei Zhao;Qingyan Niu;Jimei Su;Yanmei Huang,Shumei Ju;Yanfang Sun;Yingjie Su
URLS,,https://retractionwatch.com/2021/07/12/elsevier-says-integrity-and-rigor-of-peer-review-for-400-papers-fell-beneath-the-high-standards-expected/,https://retractionwatch.com/2021/07/12/elsevier-says-integrity-and-rigor-of-peer-review-for-400-papers-fell-beneath-the-high-standards-expected/
ArticleType,Research Article;,Research Article;,Research Article;


In [5]:
# filtering the csv to keep the retracted articles to work with 
print('RetractionNature values:')
print(df['RetractionNature'].value_counts(), '\n')

# keeping only true retracted articles 
# cleaning the data a bit
# dropping Expression of Concern, Correction, Reinstament and blank rows since they are not confirmed retractions
retractions = df[df['RetractionNature'].str.strip() == 'Retraction'].copy()

# print remaining true retractions
print('true retractions:', len(retractions))

RetractionNature values:
RetractionNature
Retraction               66727
Expression of concern     3687
Correction                1509
                           277
Reinstatement              160
Name: count, dtype: int64 

true retractions: 66727


In [7]:
# cleaning the subject labels for papers which have several subjects
subjects = (retractions['Subject']
            .str.split(';')
            .explode()
            .str.strip())

# printing top 40 subjects
print(subjects.value_counts().head(40))

Subject
                                           66727
(B/T) Technology                           12336
(BLS) Biology - Cellular                   11210
(B/T) Computer Science                      9377
(BLS) Genetics                              8497
(BLS) Biochemistry                          7660
(BLS) Biology - Cancer                      6411
(B/T) Data Science                          5755
(SOC) Education                             5243
(BLS) Biology - Molecular                   4529
(B/T) Business - Economics                  4141
(PHY) Mathematics                           4091
(HSC) Medicine - Oncology                   3852
(ENV) Environmental Sciences                3264
(PHY) Materials Science                     3140
(B/T) Business - Management                 3058
(HSC) Medicine - Pharmacology               3033
(HSC) Medicine - Surgery                    2544
(HSC) Medicine - Cardiovascular             2499
(HSC) Medicine - Neurology                  2396
(BLS) Toxico

In [13]:
# further filtering to get data related only to clinical/health and life sciences retractions
# HSC = health sciences (clinical medicine)
# BLS = biomedical 

BIOMED_PREFIXES =['(HSC)', '(BLS)']

# setting up a smaller scope (only clincal papers subset) depending on how the research is later framed 
CLINICAL_PREFIXES = '(HSC)'

# checking for biomedical subject papers
def is_biomed(subject_str):
    
    # splitting multi-subject papers into individual subjects
    subjects_split = [s.strip() for s in subject_str.split(';')] 

    # returns True if any biomed prefixes are found
    return any(p.startswith(pre) for p in subjects_split for pre in BIOMED_PREFIXES)

# same as above, filtered for ONLY clinical medicine subject papers
def is_clinical(subject_str):
    clinical_subjects_split = [s.strip() for s in subject_str.split(';')]
    return any(p.startswith(CLINICAL_PREFIXES) for p in clinical_subjects_split)

# keep only the rows with at least one biomed tag
biomed = retractions[retractions['Subject'].apply(is_biomed)].copy()

# tag the clinical biomed papers (potentially used for robustness checks later)
biomed['is_clinical'] = biomed['Subject'].apply(is_clinical)

# printing results 
print('biomed papers:', len(biomed))
print('clinical-biomed papers:', biomed['is_clinical'].sum())

biomed papers: 34019
clinical-biomed papers: 21669


In [17]:
# fetching the original paper's PubMed ID to get the abstract later
# the PubMed ID is only usable if it's a real nmber, missing ones will return as blanks or 0s
def usable_pubmed(x):
    x = str(x).strip()
    return x.isdigit() and x != '0'

# tag the biomed rows by usable PubMed ids
biomed['pubmed_id_true'] = biomed['OriginalPaperPubMedID'].apply(usable_pubmed)

# keep only the useable rows 
contains_pubmed_id = biomed[biomed['pubmed_id_true']].copy() 

# printing results
print('biomed retractions with usable PubMed IDs:', len(contains_pubmed_id))

biomed retractions with usable PubMed IDs: 26413


In [18]:
# looking at why papers were retracted (reasons)
reasons = (contains_pubmed_id['Reason']
.str.split(';')
.explode()
.str.strip())

# dropping empty strings
reasons = reasons[reasons != '']

# print top 30 retraction reasons
print(reasons.value_counts().head(30))

Reason
Investigation by Journal/Publisher                               11587
Unreliable Results and/or Conclusions                             9694
Investigation by Third Party                                      8600
Concerns/Issues about Data                                        7984
Paper Mill                                                        5215
Duplication of/in Image                                           4581
Concerns/Issues about Referencing/Attributions                    4405
Concerns/Issues about Peer Review                                 4304
Concerns/Issues about Results and/or Conclusions                  3902
Compromised Peer Review                                           2892
Computer-Aided Content or Computer-Generated Content              2779
Concerns/Issues about Image                                       2486
Investigation by Company/Institution                              2390
Original Data and/or Images not Provided and/or not Available     2110

In [21]:
# creating Tier 1 retraction reasons 
# we define these as retractions where the factuality of the science itself was compromised in some way
# we exclude procedural and suspicious tags (investigations, concerns, etc) and administrative retractions (duplication, author unresponsive, etc)

# setting the Tier 1 retraction reasons
TIER1_REASONS = [
    "Falsification/Fabrication of Data",
    "Error in Data",
    "Unreliable Data",
    "Unreliable Results and/or Conclusions",
    "Manipulation of Images",
    "Original Data and/or Images not Provided and/or not Available",
    "Misconduct by Author",
    "Misconduct - Official Investigation(s) and/or Finding(s)",]

# filtering by selected reasons 
def tier1_retraction(reason_str):
    paper_reasons = [r.strip() for r in reason_str.split(';')]
    return any(r in TIER1_REASONS for r in paper_reasons)

# tagging data
contains_pubmed_id['tier1'] = contains_pubmed_id['Reason'].apply(tier1_retraction)
tier1_pool = contains_pubmed_id[contains_pubmed_id['tier1']].copy()

# printing results
print ('tier 1 retractions:', len(tier1_pool))

tier 1 retractions: 14984


In [ ]:
# importing Biopython's interface to NCBI/PubMed to fetch the abstracts
from Bio import Entrez 
import time

Entrez.email = 'kere00003@uni-saarland.de'
Entrez.api_key = None
# function to get the abstract for a given PubMed ID
def fetch_abstract(pmid):
    try:
        
        #fetching in XML form
        handle = Entrez.efetch(db = 'pubmed', id = str(pmid), rettype = 'abstract', retmode = 'xml')
        record = Entrez.read(handle) 
        handle.close()

        # looking at XML structure to get the abstract
        article = record['PubmedArticle'][0]['MedlineCitation']['Article']

        # joining the sections
        abstract_parts = article.get('Abstract', {}).get('AbstractText', [])
        return ' '.join(str(part) for part in abstract_parts).strip()

    # no abstract
    except Exception:
        return ''

# taking 50 random articles as a test
sample = tier1_pool.sample(min(50, len(tier1_pool)), random_state = 42).copy()

# pausing between requests to not hit NCBI's rate limit (0.12s with a key = ~10/sec, 0.34s without = ~3/sec)
delay = 0.12 if Entrez.api_key else 0.34

# getting the abstract for each sample paper
abstracts, lengths = [], []
for pmid in sample['OriginalPaperPubMedID']:
    text = fetch_abstract(pmid)
    abstracts.append(text)

    # word count
    lengths.append(len(text.split()))   
    time.sleep(delay)


sample['abstract_text'] = abstracts
sample['abstract_words'] = lengths

# printing results
hit_rate = (sample['abstract_words'] > 20).mean()
print(f'abstract resolution rate: {hit_rate:.0%}')
print(f'median abstract length: {int(sample['abstract_words'].median())} words')

abstract resolution rate: 98%
median abstract length: 219 words


In [25]:
# sanity check to show any sampled papers that came back with little/no abstract
sample[sample['abstract_words'] <= 20][['OriginalPaperPubMedID', 'Title', 'abstract_words']]

,OriginalPaperPubMedID,Title,abstract_words
70684,7947511,Term Delivery After Intrauterine Relocation of...,0


In [26]:
# saving the full cleaned data 

# metadata + PubMed IDs are saved here, but not the abstracts (done later) 

# keeping the relevant columns
keep_cols = [
    'Record ID', 'Title', 'Subject', 'is_clinical', 'Journal',
    'OriginalPaperDOI', 'OriginalPaperPubMedID',
    'OriginalPaperDate', 'RetractionDate', 'Reason',]

# keeping columns that actually exist (edge case of a typo or missing column)
keep_cols = [c for c in keep_cols if c in tier1_pool.columns]

# writing to the interim data folder 
tier1_pool[keep_cols].to_csv('../data/interim/tier1_pool.csv', index = False)

print(f'saved {len(tier1_pool)} Tier-1 papers to data/interim/tier1_pool.csv')

saved 14984 Tier-1 papers to data/interim/tier1_pool.csv
